In [37]:
#Create the main Repo list
import os
import pandas as pd
from urllib.parse import urlparse

# === INPUT / OUTPUT ===
SRC_CSV = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\Clone_Status.csv"
OUT_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10"
OUT_CSV = os.path.join(OUT_DIR, "3.2_Total_Repo.csv")
os.makedirs(OUT_DIR, exist_ok=True)

# === Load ===
df = pd.read_csv(SRC_CSV, dtype=str).fillna("")
df.columns = [c.strip() for c in df.columns]

# --- Find columns ---
def pick_col(candidates, cols):
    cols_lower = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

clone_col = pick_col(["clone_status"], df.columns)
yml_col   = pick_col(["yml_detected"], df.columns)
url_col   = pick_col(["html_url"], df.columns)

if not clone_col or not yml_col or not url_col:
    missing = [name for name, col in {"clone_status": clone_col, "yml_detected": yml_col, "html_url/htm_url": url_col}.items() if not col]
    raise ValueError(f"Missing required column(s): {', '.join(missing)}")

# --- Normalize "yes" detection ---
def is_yes(x: str) -> bool:
    return str(x).strip().lower() in {"yes", "true", "y", "1"}

filtered = df[ df[clone_col].apply(is_yes) & df[yml_col].apply(is_yes) ].copy()

# --- Build full_name = owner.repo ---
def url_to_full_name(u: str) -> str:
    try:
        path = urlparse(str(u).strip()).path.strip("/")
        if not path:
            return ""
        if path.endswith(".git"):
            path = path[:-4]
        parts = path.split("/")
        if len(parts) >= 2:
            return f"{parts[0]}.{parts[1]}"
        return path
    except Exception:
        return ""

# Ensure full_name is lowercase
filtered["full_name"] = filtered[url_col].apply(url_to_full_name).str.lower()

# --- Keep only URL + full_name ---
out = filtered[[url_col, "full_name"]].rename(columns={url_col: "html_url"})

# Drop duplicates
out = out.drop_duplicates(subset=["html_url"]).reset_index(drop=True)

# Save
out.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV} (rows={len(out)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.2_Total_Repo.csv (rows=4518)


In [38]:
#Adds the Instru_tests for each repo

import os
import pandas as pd
from collections import defaultdict

# === PATHS ===
REPO_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.2_Total_Repo.csv"
TEST_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Test_Files"
OUT_CSV  = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.2_Total_Repo.csv"

# === Load repo list ===
repos = pd.read_csv(REPO_CSV, dtype=str).fillna("")
if "full_name" not in repos.columns:
    raise ValueError("Expected column 'full_name' in 3.1_Total_Repo.csv (format: owner.repo).")
repos["full_name"] = repos["full_name"].astype(str).str.strip()

# === Detection keywords ===
INSTRU_HINTS_NATIVE  = [
    "instrumentation", "androidtest", "connectedandroidtest",
    "espresso", "uiautomator", "orchestrator", "manageddevices", "gmd"
]
INSTRU_HINTS_FLUTTER = [
    "flutter", "dart"
]

def is_instru_file(fname: str) -> bool:
    name = fname.lower()
    return any(k in name for k in INSTRU_HINTS_NATIVE + INSTRU_HINTS_FLUTTER)

def classify_test(fname: str) -> str:
    lname = fname.lower()
    if any(k in lname for k in INSTRU_HINTS_FLUTTER):
        return "flutter"
    if any(k in lname for k in INSTRU_HINTS_NATIVE):
        return "native"
    return ""

# === Count tests per repo ===
native_counts  = defaultdict(int)
flutter_counts = defaultdict(int)

for fname in os.listdir(TEST_DIR):
    fpath = os.path.join(TEST_DIR, fname)
    if not os.path.isfile(fpath):
        continue
    if "__" not in fname:
        continue

    repo_token = fname.split("__", 1)[0].strip()
    if not repo_token:
        continue
    if not is_instru_file(fname):
        continue

    kind = classify_test(fname)
    if kind == "flutter":
        flutter_counts[repo_token] += 1
    elif kind == "native":
        native_counts[repo_token] += 1

# === Merge into repo DataFrame ===
repos["native_instru_test"]  = repos["full_name"].map(lambda k: native_counts.get(k, 0)).astype(int)
repos["flutter_instru_test"] = repos["full_name"].map(lambda k: flutter_counts.get(k, 0)).astype(int)
repos["Intru_test"] = (repos["native_instru_test"] + repos["flutter_instru_test"] > 0)

# === Save ===
repos.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV} (rows={len(repos)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.2_Total_Repo.csv (rows=4518)


the following cell is adjusted to includes two more new column created in yaml v4.0 for detecting flutter signal and device

In [39]:
# Aggregate instru_t_ci_signal and per-platform YML counts, append to main (case-insensitive)
import os
import re
import pandas as pd

BASE_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10"
MAIN_CSV = os.path.join(BASE_DIR, "3.2_Total_Repo.csv")
YML_CSV  = os.path.join(BASE_DIR, "3.1.1_YML_Files_V4.0.csv")
OUT_CSV  = MAIN_CSV  # overwrite

def normalize_name(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.lower()

def sanitize_col(name: str) -> str:
    s = re.sub(r"\W+", "_", str(name).strip().lower())
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "unknown"

def to_bool_series(s: pd.Series) -> pd.Series:
    truthy = {"true","1","yes","y","t"}
    falsy  = {"false","0","no","n","f"}
    s = s.astype(str).str.strip().str.lower()
    return s.map(lambda x: True if x in truthy else (False if x in falsy else False)).astype("boolean")

def dedup_preserve(seq):
    seen, out = set(), []
    for x in seq:
        x = str(x)
        if x and x not in seen:
            seen.add(x); out.append(x)
    return out

# --- Load main ---
if not os.path.isfile(MAIN_CSV):
    raise FileNotFoundError(f"Main CSV not found: {MAIN_CSV}")
main = pd.read_csv(MAIN_CSV, dtype=str).fillna("")
if "full_name" not in main.columns:
    raise ValueError("Main CSV must contain 'full_name'.")
main["full_name_lc"] = normalize_name(main["full_name"])

# --- Load YML ---
if not os.path.isfile(YML_CSV):
    raise FileNotFoundError(f"YML CSV not found: {YML_CSV}")
yml = pd.read_csv(YML_CSV, dtype=str).fillna("")
for col in ("full_name", "ci_platform"):
    if col not in yml.columns:
        raise ValueError(f"{os.path.basename(YML_CSV)} must contain '{col}'.")

# ensure expected columns exist
for col, default in (
    ("instru_t_ci_signal", ""),
    ("confidence", ""),
    ("confidence_reason", ""),
    # NEW: Flutter columns expected from detector; create safe defaults if absent
    ("flutter_integ_t_signal", "false"),
    ("flutter_integ_t_d", ""),
):
    if col not in yml.columns:
        yml[col] = default

yml["full_name_lc"] = normalize_name(yml["full_name"])

# --- Aggregate per-platform YML counts and totals ---
plat_counts = (
    yml.groupby(["full_name_lc", "ci_platform"])
       .size()
       .unstack(fill_value=0)
)
if not plat_counts.empty:
    plat_counts = plat_counts.rename(columns={c: sanitize_col(c) for c in plat_counts.columns})
    # collapse potential duplicates caused by sanitation
    plat_counts = plat_counts.T.groupby(level=0).sum().T
    plat_counts = plat_counts.reset_index()
else:
    plat_counts = pd.DataFrame(columns=["full_name_lc"])

total_counts = (
    yml.groupby("full_name_lc").size().rename("Total_YMLs").reset_index()
)

# --- Aggregate instru_t_ci_signal (True if any row True) ---
yml["instru_t_ci_signal"] = to_bool_series(yml["instru_t_ci_signal"])
agg_flag = (
    yml[["full_name_lc", "instru_t_ci_signal"]]
      .groupby("full_name_lc", as_index=False)["instru_t_ci_signal"]
      .any()
)
agg_flag["instru_t_ci_signal"] = agg_flag["instru_t_ci_signal"].astype("boolean")

# --- Aggregate confidence + confidence_reason per repo ---
yml["conf_norm"] = yml["confidence"].str.strip().str.lower()
def aggregate_confidence(group: pd.DataFrame) -> pd.Series:
    confs = set(group["conf_norm"].dropna().tolist())
    if "high" in confs:
        level = "high"
    elif "medium" in confs:
        level = "medium"
    elif "low" in confs:
        level = "low"
    else:
        level = ""

    if level:
        reasons = group.loc[group["conf_norm"] == level, "confidence_reason"].astype(str).str.strip()
        reasons = [r for r in reasons if r]
        agg_reason = " || ".join(dedup_preserve(reasons))
    else:
        agg_reason = ""

    return pd.Series({"instru_t_ci_confidence": level, "confidence_reason": agg_reason})

agg_conf = (
    yml.groupby("full_name_lc")
       .apply(aggregate_confidence)
       .reset_index()
)

# --- NEW: Aggregate Flutter columns ---
# repo-level flutter_integ_t_signal: True if any row True
yml["flutter_integ_t_signal"] = to_bool_series(yml["flutter_integ_t_signal"])
flutter_sig = (
    yml.groupby("full_name_lc", as_index=False)["flutter_integ_t_signal"]
       .any()
)
flutter_sig["flutter_integ_t_signal"] = flutter_sig["flutter_integ_t_signal"].astype("boolean")

# repo-level flutter_integ_t_d: comma-separated unique list (lowercased)
yml["flutter_integ_t_d_norm"] = yml["flutter_integ_t_d"].astype(str).str.strip().str.lower()
def agg_flutter_devices(group: pd.DataFrame) -> pd.Series:
    vals = [v for v in group["flutter_integ_t_d_norm"].tolist() if v and v not in {"nan","none"}]
    vals = dedup_preserve(vals)
    return pd.Series({"flutter_integ_t_d": ", ".join(vals)})

flutter_dev = (
    yml.groupby("full_name_lc")
       .apply(agg_flutter_devices)
       .reset_index()
)

# --- Merge aggregates together ---
agg = (total_counts
       .merge(plat_counts, on="full_name_lc", how="left")
       .merge(agg_flag,    on="full_name_lc", how="left")
       .merge(agg_conf,    on="full_name_lc", how="left")
       .merge(flutter_sig, on="full_name_lc", how="left")    # NEW
       .merge(flutter_dev, on="full_name_lc", how="left"))   # NEW

# --- Merge into main (case-insensitive) ---
out = main.merge(agg, on="full_name_lc", how="left").drop(columns=["full_name_lc"])

# Normalize numeric/platform columns to int (fill NaN with 0)
platform_cols = [c for c in plat_counts.columns if c != "full_name_lc"]
for c in platform_cols:
    if c not in out.columns:
        out[c] = 0
    out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0).astype(int)

if "Total_YMLs" not in out.columns:
    out["Total_YMLs"] = 0
out["Total_YMLs"] = pd.to_numeric(out["Total_YMLs"], errors="coerce").fillna(0).astype(int)

# Final instru_t_ci_signal in main: aggregated value
existing = out["instru_t_ci_signal"] if "instru_t_ci_signal" in out.columns else pd.Series([False]*len(out), index=out.index)
existing = to_bool_series(existing).fillna(False)
yml_flag = out.get("instru_t_ci_signal", pd.Series([False]*len(out), index=out.index))
yml_flag = to_bool_series(yml_flag).fillna(False)
out["instru_t_ci_signal"] = yml_flag.astype("boolean")

# --- NEW: Ensure Flutter columns exist with correct types/defaults ---
if "flutter_integ_t_signal" not in out.columns:
    out["flutter_integ_t_signal"] = False
out["flutter_integ_t_signal"] = to_bool_series(out["flutter_integ_t_signal"]).fillna(False).astype("boolean")

if "flutter_integ_t_d" not in out.columns:
    out["flutter_integ_t_d"] = ""
out["flutter_integ_t_d"] = out["flutter_integ_t_d"].astype(str).fillna("").str.strip()

# Save
out.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV} (rows={len(out)})")
print(f"instru_t_ci_signal True={int(out['instru_t_ci_signal'].sum())} / {len(out)}")
print("Platform columns added:", [c for c in platform_cols if c in out.columns])
print(f"flutter_integ_t_signal True={int(out['flutter_integ_t_signal'].sum())} / {len(out)}")


C:\Users\gilla\AppData\Local\Temp\ipykernel_15396\6365525.py:114: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(aggregate_confidence)


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.2_Total_Repo.csv (rows=4518)
instru_t_ci_signal True=449 / 4518
Platform columns added: ['appveyor', 'azure_pipelines', 'bitbucket', 'bitrise', 'circle_ci', 'cirrus', 'codemagic', 'github_actions', 'gitlab', 'semaphore', 'travis_ci']
flutter_integ_t_signal True=35 / 4518


C:\Users\gilla\AppData\Local\Temp\ipykernel_15396\6365525.py:136: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(agg_flutter_devices)


In [40]:
import os
import pandas as pd

BASE_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10"

FILES = {
    "instru_t_signal_config": ("3.1.1_Instru_T_Signal_Config.csv",
                               ["instru_t_signal_config", "instru_test_signal_config"]),
    "unit_t_signal_ci":       ("3.1.2_Unit_T_Signal_CI.csv",
                               ["unit_t_signal_ci", "unit_test_signal_ci"]),
    "unit_t_signal_config":   ("3.1.2_Unit_T_Signal_Config.csv",
                               ["unit_t_signal_config", "unit_test_config_signal"]),
}

MAIN_CSV = os.path.join(BASE_DIR, "3.2_Total_Repo.csv")
OUT_CSV  = MAIN_CSV  # overwrite

def normalize_name(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.lower()

def to_bool_series(s: pd.Series) -> pd.Series:
    truthy = {"true","1","yes","y","t"}
    falsy  = {"false","0","no","n","f"}
    s = s.astype(str).str.strip().str.lower()
    return s.map(lambda x: True if x in truthy else (False if x in falsy else False)).astype("boolean")

def load_and_aggregate_bool(path: str, out_col: str, candidates: list[str]) -> pd.DataFrame:
    if not os.path.isfile(path):
        print(f"[WARN] Missing file -> {os.path.basename(path)}; skipping {out_col}.")
        return pd.DataFrame(columns=["full_name", out_col])
    df = pd.read_csv(path, dtype=str).fillna("")
    if "full_name" not in df.columns:
        print(f"[WARN] 'full_name' not in {os.path.basename(path)}; skipping {out_col}.")
        return pd.DataFrame(columns=["full_name", out_col])

    df["full_name"] = normalize_name(df["full_name"])
    flag_col = next((c for c in candidates if c in df.columns), None)
    if flag_col is None:
        print(f"[WARN] None of {candidates} found in {os.path.basename(path)}; skipping {out_col}.")
        return pd.DataFrame(columns=["full_name", out_col])

    b = to_bool_series(df[flag_col])
    agg = (
        pd.DataFrame({"full_name": df["full_name"], out_col: b})
          .groupby("full_name", as_index=False)[out_col]
          .any()
    )
    agg[out_col] = agg[out_col].astype("boolean")
    return agg

# --- Load & normalize main ---
if not os.path.isfile(MAIN_CSV):
    raise FileNotFoundError(f"Main CSV not found: {MAIN_CSV}")

main = pd.read_csv(MAIN_CSV, dtype=str).fillna("")
if "full_name" not in main.columns:
    raise ValueError("Main CSV must contain a 'full_name' column.")
main["full_name"] = normalize_name(main["full_name"])

# --- Merge aggregated flags (default False; True if any repo hit) ---
for out_col, (fname, aliases) in FILES.items():
    agg_path = os.path.join(BASE_DIR, fname)
    agg = load_and_aggregate_bool(agg_path, out_col, aliases)

    # ensure column exists in main with default False (nullable boolean)
    if out_col not in main.columns:
        main[out_col] = pd.Series([False] * len(main), index=main.index, dtype="boolean")
    else:
        # coerce any existing values safely -> boolean, unknowns -> False
        main[out_col] = to_bool_series(main[out_col])

    if not agg.empty:
        lookup = agg.set_index("full_name")[out_col]
        upd = main["full_name"].map(lookup).astype("boolean").fillna(False)
        # OR (True wins)
        main[out_col] = (main[out_col] | upd).astype("boolean")

# --- Save ---
main.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV} (rows={len(main)})")

# Optional: quick counts
for col in FILES.keys():
    if col in main.columns:
        print(f"{col}: True={int(main[col].sum())} / {len(main)}")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.2_Total_Repo.csv (rows=4518)
instru_t_signal_config: True=2280 / 4518
unit_t_signal_ci: True=2162 / 4518
unit_t_signal_config: True=2303 / 4518


In [41]:
# -*- coding: utf-8 -*-
# Left-join metadata onto main without bringing duplicate column names or excluded metadata fields.

import os
import pandas as pd
from datetime import datetime
from urllib.parse import urlparse

BASE_DIR   = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10"
MAIN_PATH  = os.path.join(BASE_DIR, "3.2_Total_Repo.csv")
META_PATH  = os.path.join(BASE_DIR, "Project_Metadata.csv")
OUT_PATH   = MAIN_PATH  # overwrite

FETCH_DATE = datetime(2025, 8, 10)

# Columns from metadata to explicitly exclude
EXCLUDE_META_COLS = {
    "html_url", "repo_index", "repo_name", "id", "name", "full_name", "owner"
}

def norm_key_main(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
              .str.replace("/", ".", regex=False)
              .str.strip()
              .str.lower()
    )

def owner_repo_from_html_url(url: str) -> tuple[str, str]:
    try:
        path = urlparse(str(url)).path.strip("/")
        parts = path.split("/")
        if len(parts) >= 2:
            return parts[0].strip(), parts[1].strip()
    except Exception:
        pass
    return "", ""

def meta_key_from_html(series: pd.Series) -> pd.Series:
    return series.astype(str).map(lambda u: ".".join(owner_repo_from_html_url(u))).str.lower()

# --- Load ---
main = pd.read_csv(MAIN_PATH, dtype=str).fillna("")
meta = pd.read_csv(META_PATH, dtype=str).fillna("")

# --- Validate ---
if "full_name" not in main.columns:
    raise ValueError("Main file must contain a 'full_name' column.")
if "html_url" not in meta.columns:
    raise ValueError("Metadata file must contain an 'html_url' column.")

# --- Keys ---
main["__key__"] = norm_key_main(main["full_name"])
meta["__key__"] = meta_key_from_html(meta["html_url"])

# --- repo_age ---
if "created_at" in meta.columns:
    created_dt = pd.to_datetime(meta["created_at"], errors="coerce", utc=True)
    meta["repo_age"] = ((pd.to_datetime(FETCH_DATE) - created_dt.dt.tz_localize(None)).dt.days / 365.25).round(2)
else:
    meta["repo_age"] = ""

# --- Filter metadata columns ---
main_cols_set = set(main.columns)
meta_cols_to_add = [
    c for c in meta.columns
    if c not in main_cols_set and c != "__key__" and c not in EXCLUDE_META_COLS
]

meta_to_merge = meta[["__key__"] + meta_cols_to_add]

# --- Merge ---
out = main.merge(meta_to_merge, on="__key__", how="left").drop(columns=["__key__"], errors="ignore")

# --- Save ---
out.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH} (rows={len(out)}, metadata_cols_added={len(meta_cols_to_add)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.2_Total_Repo.csv (rows=4518, metadata_cols_added=30)
